# arange-fancy-index-cross-entropy — ex2: full mean cross-entropy loss via logsumexp − picked

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `arange-fancy-index-cross-entropy`. Running the final beacon cell reports progress against the `Loss: arange fancy-index cross-entropy` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Loss: arange fancy-index cross-entropy` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arange-fancy-index-cross-entropy`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arange-fancy-index-cross-entropy"
DD_SUBTOPIC = "Loss: arange fancy-index cross-entropy"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Full mean cross-entropy via `logsumexp - picked` — quick refresher

Per-sample CE on logits `(B, C)` with integer targets `(B,)`:

```
lse_i   = log sum_c exp(logits[i, c])            # logsumexp over class axis
picked_i = logits[i, target[i]]                  # arange fancy-index
ce_i    = lse_i - picked_i                       # = -log softmax(logits)[i, target[i]]
loss    = mean_i(ce_i)                           # scalar
```

Why `lse - picked` IS cross-entropy. `softmax(logits)[i, c] = exp(logits[i, c]) / sum_c' exp(logits[i, c'])`. Negate the log, expand, and the denominator becomes `lse_i`, the numerator becomes `picked_i`.

**Exemplar.** `logits = [[1.0, 1.0, 1.0]]`, `target = [0]`. `lse = log(3 * e^1) = 1 + log 3 ~= 2.0986`, `picked = 1.0`, `ce ~= 1.0986 == log 3` (uniform-over-3 CE).

Matches `F.cross_entropy(logits, target, reduction='mean')` to fp tol.

### Exercise 2 — full mean cross-entropy loss via logsumexp − picked

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-sample CE decomposition `ce_i = lse_i - logits[i, target[i]]` and average over the batch axis to produce a scalar loss matching `F.cross_entropy(reduction='mean')`.
> Keywords: cross-entropy, logsumexp, arange-fancy-index, mean-reduce
> ```

**KCs targeted:** `arange-fancy-index-cross-entropy`, `logsumexp-cross-entropy`

Implement `ex2_cross_entropy_mean(logits, target)`. Returns a scalar tensor equal to `F.cross_entropy(logits, target, reduction='mean')`.

Inputs:
- `logits`: shape `(B, C)`, float.
- `target`: shape `(B,)`, integer class indices in `[0, C)`.

**Recipe (three composable steps — all vectorized).**
```
lse_per_sample = t.logsumexp(logits, dim=-1)        # (B,)
picked         = logits[t.arange(B), target]        # (B,)  — ex1 facet
ce_per_sample  = lse_per_sample - picked            # (B,)
loss           = ce_per_sample.mean()               # scalar
```

**Why the formula is correct.** `-log softmax(logits)[i, c] = -log(exp(logits[i, c]) / sum_c' exp(logits[i, c'])) = logsumexp_c'(logits[i, c']) - logits[i, c]`. So the standard `NLL(log_softmax(logits))[i] = lse_i - picked_i`. Mean over `i` is the canonical `reduction='mean'` form.

**Forbidden:** calling `F.cross_entropy` or `F.nll_loss` directly — the drill is the build-from-pieces version. `t.logsumexp` and fancy-indexing are OK (they're the primitives we're composing).

**Output is a 0-D scalar tensor**, not a Python float. `.shape == ()`.

In [ ]:
def ex2_cross_entropy_mean(logits: Tensor, target: Tensor) -> Tensor:
    B = logits.shape[0]
    lse    = t.logsumexp(logits, dim=-1)             # (B,)
    picked = logits[t.arange(B), target]             # (B,)
    return (lse - picked).mean()                     # scalar


<details><summary>Solution</summary>

```python
def ex2_cross_entropy_mean(logits: Tensor, target: Tensor) -> Tensor:
    B = logits.shape[0]
    lse    = t.logsumexp(logits, dim=-1)             # (B,)
    picked = logits[t.arange(B), target]             # (B,)
    return (lse - picked).mean()                     # scalar
```

**The 3-line CE is a teaching artifact.** Real `F.cross_entropy` fuses these into a single CUDA kernel with label smoothing, ignore_index handling, weighted-mean reduction, and gradient bookkeeping. But the math is EXACTLY these three steps. When you debug a custom loss (reweighting per-sample, masking, hierarchical softmax), you'll be working at this decomposed level.

**Why `t.logsumexp`, not `(logits.exp()).sum(-1).log()`.** Numerical stability. `t.logsumexp` subtracts the per-row max before the exp, preventing overflow at large logits. The test for `1000`-scale logits checks this — a naive implementation would return `inf` or `nan` and crash the comparison.

**`.mean()` vs `.sum() / B`.** Both give the same scalar; `.mean()` keeps the dtype's natural epsilon. For tiny batches `(B=1, 2)` the difference is irrelevant, but at fp16 with large batches the in-kernel mean is more accurate.

**Composes with the autograd preamble.** The MiniTensor scaffolding is loaded so this drill can sit next to other autograd-internals drills, but THIS exercise uses raw `torch.Tensor` for clarity — same code translates directly to `MiniTensor` if you wrap each op with its `_back` fn.

**Composes with grads-dict-accumulate.** When you differentiate this loss by hand: the gradient w.r.t. `logits[i, c]` is `softmax(logits)[i, c] - (1 if c == target[i] else 0)`, divided by `B`. That's the canonical 'softmax minus one-hot' gradient that every classification head uses on the backward pass.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()